# Analysis of Semantic Image Communication Results (v1)

Loads `../results/experiment_results.csv` (written by `experiment.py`) and compares the **semantic** pipeline against the **JPEG-matched** and **text-only** baselines on payload size, PSNR, and downstream detector recall.

Requires `pandas`, `seaborn`, `matplotlib` (see `requirements-extra.txt`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load the results table

One row per (image, method); methods are `semantic`, `jpeg_matched`, `text_only`.

In [ ]:
df = pd.read_csv("../results/experiment_results.csv")
df.head(12)

## 2. Per-method means

Average the key metrics for each method across all images.

In [ ]:
summary = df.groupby("method").agg(
    compression_ratio=("compression_ratio", "mean"),
    psnr=("psnr", "mean"),
    downstream_class_recall=("downstream_class_recall", "mean"),
    payload_bytes=("payload_bytes", "mean"),
).reset_index()
summary

## 3. PSNR and downstream recall by method

PSNR favors JPEG (it reconstructs every pixel), but the semantic pipeline keeps objects detectable; the text-only baseline (no crops) collapses on downstream recall — showing the crops matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=summary, x="method", y="psnr", ax=axes[0], color="steelblue")
axes[0].set_title("Mean PSNR by method")
axes[0].set_ylabel("PSNR (dB)")
sns.barplot(data=summary, x="method", y="downstream_class_recall", ax=axes[1], color="indianred")
axes[1].set_title("Mean downstream class recall by method")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Payload size by method

The semantic and JPEG-matched payloads are budget-matched by construction; text-only is far smaller because it sends no crops.

In [ ]:
plt.figure()
sns.barplot(data=summary, x="method", y="payload_bytes", color="seagreen")
plt.title("Mean payload size by method")
plt.ylabel("Payload (bytes)")
plt.tight_layout()
plt.show()